# 🧪 Notebook 02: Model Training, Optuna HPO & Comparative Benchmarking

**Conference:** 2026 IEEE 8th International Conference 'Actual Problems of Unmanned Aerial Vehicles Development' (APUAVD-2026)
**Author:** Yaroslav Fetisov (NTUU 'KPI')

This notebook benchmarks **DASU-Net** (Dual-Attention Swin U-Net with PPNMM Non-Linear Decoder) against **DeepTrans-HSU Baseline** (ViT + Linear) on UAV hyperspectral flight data.


In [ ]:
import sys
sys.path.append('..')
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.data.dataset import HyperspectralDataset
from src.models.unmixer import DASUNet
from src.core.losses import TotalLoss
from src.core.metrics import compute_rmse, compute_sad, match_endmembers

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 1. Initialize Dataset & Model
DASU-Net combines a Shifted-Window Swin Transformer Encoder with a Dual-Attention Block (Spatial + XCA Spectral) and a PPNMM Non-Linear Decoder.


In [ ]:
dataset = HyperspectralDataset('uav_synthetic', data_dir='../data/raw', device=device)
model = DASUNet(
    num_endmembers=dataset.P,
    num_bands=dataset.L,
    spatial_size=dataset.col,
    encoder_type='swin',
    decoder_type='nonlinear',
    use_dual_attention=True,
    nonlinear_gamma=0.45
).to(device)
model.apply(model.weights_init)
model.init_decoder_weights(dataset.get_image_cube(), method='sivm')
print(model)


## 2. Train DASU-Net Model


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=4e-3, weight_decay=4e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.8)
loss_fn = TotalLoss(num_bands=dataset.L, beta=2500, gamma=0.015, delta=5e-4, lambda_reg=5e-4).to(device)
clipper = model.get_clipper()
img_cube = dataset.get_image_cube()

losses = []
model.train()
for epoch in range(150):
    abu, recon = model(img_cube)
    endmem = model.decoder.get_endmembers()
    loss = loss_fn(recon, img_cube, abu, endmem)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0, norm_type=1)
    optimizer.step()
    model.decoder.apply(clipper)
    scheduler.step()
    losses.append(loss.item())
    if (epoch + 1) % 30 == 0:
        print(f'Epoch {epoch+1}/150 - Loss: {loss.item():.4f}')


## 3. Evaluation & Hungarian Matching


In [ ]:
model.eval()
with torch.no_grad():
    abu, recon = model(img_cube)
abu = abu / abu.sum(dim=1, keepdim=True).clamp(min=1e-8)
abu_np = abu.squeeze(0).permute(1, 2, 0).cpu().numpy()
target_np = dataset.get_abundance_cube().cpu().numpy()
est_endmem = model.decoder.get_endmembers().cpu().numpy()
true_endmem = dataset.get_endmembers().numpy()

est_endmem, abu_np, perm = match_endmembers(est_endmem, true_endmem, abu_np, target_np)
rmse_cls, rmse_mean = compute_rmse(abu_np, target_np)
sad_cls, sad_mean = compute_sad(est_endmem, true_endmem)

print(f'=== Evaluation Results ===')
print(f'Mean RMSE: {rmse_mean:.4f}')
print(f'Mean SAD:  {sad_mean:.4f} rad')
